<a href="https://colab.research.google.com/github/rsvedajanaani2005-art/Code-authorship-identification/blob/main/Semeval_subtask_b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers accelerate


In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader

from torch.optim import AdamW  # <-- FIXED
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

from sklearn.metrics import f1_score


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# TODO: CHANGE THIS to your folder where the .parquet files live
BASE_DIR = "/content/drive/MyDrive/"  # <-- edit

train_path = os.path.join(BASE_DIR, "train.parquet")
val_path   = os.path.join(BASE_DIR, "validation.parquet")
test_path  = os.path.join(BASE_DIR, "test.parquet")

train_df = pd.read_parquet(train_path)
val_df   = pd.read_parquet(val_path)
test_df  = pd.read_parquet(test_path)

print("Train:", train_df.shape, train_df.columns.tolist())
print("Val:", val_df.shape, val_df.columns.tolist())
print("Test:", test_df.shape, test_df.columns.tolist())

print("\nTrain label distribution:")
print(train_df["label"].value_counts().sort_index())


Mounted at /content/drive
Train: (500000, 4) ['code', 'generator', 'label', 'language']
Val: (100000, 4) ['code', 'generator', 'label', 'language']
Test: (500000, 2) ['ID', 'code']

Train label distribution:
label
0     442096
1       4162
2       8993
3       3029
4       2227
5       1968
6       5783
7       8197
8       8127
9       4608
10     10810
Name: count, dtype: int64


In [ ]:
# Split majority vs minority
df_majority = train_df[train_df["label"] == 0]
df_minority = train_df[train_df["label"] != 0]

print("Majority (0) count:", len(df_majority))
print("Minority (!=0) count:", len(df_minority))

KEEP_MAJ = 80000  # you can try 100000 later

rng = np.random.default_rng(42)
idx_majority = rng.choice(df_majority.index.values, size=KEEP_MAJ, replace=False)
df_majority_sampled = df_majority.loc[idx_majority]

train_balanced_df = pd.concat(
    [df_majority_sampled, df_minority],
    ignore_index=True
).sample(frac=1.0, random_state=42).reset_index(drop=True)

print("Balanced train size:", len(train_balanced_df))
print("Balanced label distribution:")
print(train_balanced_df["label"].value_counts().sort_index())


Majority (0) count: 442096
Minority (!=0) count: 57904
Balanced train size: 137904
Balanced label distribution:
label
0     80000
1      4162
2      8993
3      3029
4      2227
5      1968
6      5783
7      8197
8      8127
9      4608
10    10810
Name: count, dtype: int64


In [ ]:
MAX_CHARS_TRANS = 800  # safe for tokens

def make_text_for_transformer(df):
    if "language" in df.columns:
        lang = df["language"].fillna("")
    else:
        lang = pd.Series([""] * len(df), index=df.index)
    code = df["code"].fillna("").str.slice(0, MAX_CHARS_TRANS)
    return (lang + " " + code).astype(str).tolist()

train_texts = make_text_for_transformer(train_balanced_df)
train_labels = train_balanced_df["label"].to_numpy().astype(int)

val_texts = make_text_for_transformer(val_df)
val_labels = val_df["label"].to_numpy().astype(int)

num_labels = len(np.unique(train_labels))
print("Num labels:", num_labels)
print("Label set:", np.unique(train_labels))


Num labels: 11
Label set: [ 0  1  2  3  4  5  6  7  8  9 10]


In [ ]:
MODEL_NAME = "microsoft/codebert-base"
MAX_TOKENS = 256
BATCH_SIZE = 8   # safe for T4 GPU

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class CodeDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = int(self.labels[idx])

        enc = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )

        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item

train_dataset = CodeDataset(train_texts, train_labels, tokenizer, max_length=MAX_TOKENS)
val_dataset   = CodeDataset(val_texts,   val_labels,   tokenizer, max_length=MAX_TOKENS)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

len(train_dataset), len(val_dataset)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

(137904, 100000)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels
)
model.to(device)

EPOCHS = 3  # start with 2; if stable, you can try 3

optimizer = AdamW(model.parameters(), lr=2e-5)

total_steps = len(train_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

scaler = torch.cuda.amp.GradScaler()


Using device: cuda


pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/codebert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

/tmp/ipython-input-790851007.py:22: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [ ]:
def evaluate(model, data_loader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in data_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            labels = batch["labels"]

            with torch.cuda.amp.autocast():
                outputs = model(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    labels=labels
                )
            logits = outputs.logits
            preds = torch.argmax(logits, dim=-1)

            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return macro_f1


In [ ]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    print(f"\n===== Transformer Epoch {epoch+1}/{EPOCHS} =====")
    for step, batch in enumerate(tqdm(train_loader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch["labels"]

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=labels
            )
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()

        if (step + 1) % 200 == 0:
            avg_loss = total_loss / (step + 1)
            print(f"Step {step+1}/{len(train_loader)}, loss={avg_loss:.4f}")

    avg_train_loss = total_loss / len(train_loader)
    val_macro_f1 = evaluate(model, val_loader)
    print(f"Epoch {epoch+1}: train_loss={avg_train_loss:.4f}, val_macro_f1={val_macro_f1:.4f}")



===== Transformer Epoch 1/3 =====


  0%|          | 0/17238 [00:00<?, ?it/s]

/tmp/ipython-input-547112942.py:12: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Step 200/17238, loss=2.3608
Step 400/17238, loss=2.0985
Step 600/17238, loss=1.9254
Step 800/17238, loss=1.8263
Step 1000/17238, loss=1.7458
Step 1200/17238, loss=1.6692
Step 1400/17238, loss=1.6126
Step 1600/17238, loss=1.5668
Step 1800/17238, loss=1.5276
Step 2000/17238, loss=1.4885
Step 2200/17238, loss=1.4497
Step 2400/17238, loss=1.4179
Step 2600/17238, loss=1.3864
Step 2800/17238, loss=1.3634
Step 3000/17238, loss=1.3427
Step 3200/17238, loss=1.3260
Step 3400/17238, loss=1.3095
Step 3600/17238, loss=1.2910
Step 3800/17238, loss=1.2755
Step 4000/17238, loss=1.2608
Step 4200/17238, loss=1.2454
Step 4400/17238, loss=1.2329
Step 4600/17238, loss=1.2207
Step 4800/17238, loss=1.2099
Step 5000/17238, loss=1.1984
Step 5200/17238, loss=1.1864
Step 5400/17238, loss=1.1787
Step 5600/17238, loss=1.1712
Step 5800/17238, loss=1.1620
Step 6000/17238, loss=1.1542
Step 6200/17238, loss=1.1468
Step 6400/17238, loss=1.1377
Step 6600/17238, loss=1.1297
Step 6800/17238, loss=1.1243
Step 7000/17238, l

/tmp/ipython-input-160491599.py:11: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


...

  0%|          | 0/17238 [00:00<?, ?it/s]

/tmp/ipython-input-547112942.py:12: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Step 200/17238, loss=0.5770
Step 400/17238, loss=0.5540
Step 600/17238, loss=0.5717
Step 800/17238, loss=0.5613
Step 1000/17238, loss=0.5615
Step 1200/17238, loss=0.5634
Step 1400/17238, loss=0.5619
Step 1600/17238, loss=0.5591
Step 1800/17238, loss=0.5602
Step 2000/17238, loss=0.5603
Step 2200/17238, loss=0.5616
Step 2400/17238, loss=0.5614
Step 2600/17238, loss=0.5583
Step 2800/17238, loss=0.5569
Step 3000/17238, loss=0.5554
Step 3200/17238, loss=0.5547
Step 3400/17238, loss=0.5577
Step 3600/17238, loss=0.5574
Step 3800/17238, loss=0.5564
Step 4000/17238, loss=0.5552
Step 4200/17238, loss=0.5533
Step 4400/17238, loss=0.5516
Step 4600/17238, loss=0.5508
Step 4800/17238, loss=0.5499
Step 5000/17238, loss=0.5498
Step 5200/17238, loss=0.5500
Step 5400/17238, loss=0.5520
Step 5600/17238, loss=0.5519
Step 5800/17238, loss=0.5518
Step 6000/17238, loss=0.5506
Step 6200/17238, loss=0.5508
Step 6400/17238, loss=0.5522
Step 6600/17238, loss=0.5523
Step 6800/17238, loss=0.5516
Step 7000/17238, l

/tmp/ipython-input-160491599.py:11: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 3: train_loss=0.5390, val_macro_f1=0.5004


In [ ]:
save_dir = os.path.join(BASE_DIR, "codebert_attempt_final")

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print("Saved model to:", save_dir)


Saved model to: /content/drive/MyDrive/codebert_attempt_final


In [ ]:
# 1. Prepare test texts
test_texts = make_text_for_transformer(test_df)  # uses code + (no language)

# Dummy labels (not used, but Dataset expects them)
dummy_labels = np.zeros(len(test_texts), dtype=int)

test_dataset = CodeDataset(test_texts, dummy_labels, tokenizer, max_length=MAX_TOKENS)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


In [ ]:
# 2. Run inference
model.eval()
all_test_preds = []

with torch.no_grad():
    for batch in tqdm(test_loader):
        # remove labels if present
        batch = {k: v.to(device) for k, v in batch.items() if k != "labels"}
        with torch.cuda.amp.autocast():
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"]
            )
        logits = outputs.logits
        preds = torch.argmax(logits, dim=-1)
        all_test_preds.append(preds.cpu().numpy())

all_test_preds = np.concatenate(all_test_preds)


  0%|          | 0/62500 [00:00<?, ?it/s]

/tmp/ipython-input-4160591115.py:9: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


In [ ]:
# 3. Build submission
submission = pd.DataFrame({
    "ID": test_df["ID"],                 # test column is 'ID'
    "label": all_test_preds.astype(int)  # labels 0..10
})

sub_path = os.path.join(BASE_DIR, "submission_attempt_final.csv")
submission.to_csv(sub_path, index=False)
sub_path


'/content/drive/MyDrive/submission_attempt_final.csv'

In [ ]:
test_sample_path = os.path.join(BASE_DIR, "test_sample.parquet")
test_sample_df = pd.read_parquet(test_sample_path)

print("test_sample shape:", test_sample_df.shape)
print(test_sample_df.columns)
print(test_sample_df["label"].value_counts().sort_index())


test_sample shape: (1000, 4)
Index(['code', 'generator', 'label', 'language'], dtype='object')
label
0     474
1      21
2      73
3      21
4      10
5      36
6      54
7      61
8      18
9      18
10    214
Name: count, dtype: int64


In [ ]:
test_sample_texts = make_text_for_transformer(test_sample_df)
test_sample_labels = test_sample_df["label"].to_numpy().astype(int)


In [ ]:
test_sample_dataset = CodeDataset(
    test_sample_texts,
    test_sample_labels,
    tokenizer,
    max_length=MAX_TOKENS
)

test_sample_loader = DataLoader(
    test_sample_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


In [ ]:
test_sample_macro_f1 = evaluate(model, test_sample_loader)
print("Macro F1 on test_sample:", test_sample_macro_f1)


/tmp/ipython-input-160491599.py:11: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Macro F1 on test_sample: 0.32057935013275646
